# fine-tune-forge — QLoRA Fine-tuning Pipeline

Pipeline per il fine-tuning di piccoli LLM su task verticali.

**Runtime consigliato**: `Runtime → Change runtime type → T4 GPU`

**Step**:
1. Setup (installazione + mount Drive)
2. Configurazione API keys
3. Generazione dataset con Gemini
4. Training QLoRA
5. Export (HuggingFace Hub / GGUF)

## 1. Setup

In [ ]:
# Controlla GPU disponibile
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else '⚠️  Nessuna GPU — vai su Runtime → Change runtime type → T4 GPU')

In [ ]:
# Mount Google Drive (per salvare modello e dataset)
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/fine-tune-forge'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Drive montato → {DRIVE_DIR}')

In [ ]:
# Clona il repo
!git clone https://github.com/MatteoAdamo82/fine-tuning.git /content/fine-tuning
%cd /content/fine-tuning

In [ ]:
# Installa dipendenze
!pip install -q -e . 2>&1 | tail -5
print('✓ Dipendenze installate')

## 2. Configurazione

In [ ]:
# Imposta API keys (usa Colab Secrets oppure inserisci qui)
# Consigliato: Runtime → Secrets (icona chiave) → aggiungi GEMINI_API_KEY e HF_TOKEN
try:
    from google.colab import userdata
    os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('✓ API keys caricate da Colab Secrets')
except Exception:
    # Fallback: inserisci manualmente
    import getpass
    os.environ['GEMINI_API_KEY'] = getpass.getpass('GEMINI_API_KEY: ')
    os.environ['HF_TOKEN'] = getpass.getpass('HF_TOKEN (lascia vuoto se non usi HF Hub): ')

In [ ]:
# Parametri del run — modifica qui
DOMINIO   = 'restaurant_booking'   # restaurant_booking | sales_agent | professional_booking
MODELLO   = 'Qwen/Qwen3-0.6B'     # Qwen/Qwen3-0.6B (demo 2GB) | Qwen/Qwen3-4B (6GB)
N_ESEMPI  = 50                     # esempi dataset (50 per test veloce, 150 per produzione)
ESPORTA_HF = False                 # True per pushare su HuggingFace Hub
HF_REPO   = ''                     # es. 'MatteoAdamo82/agente-ristorante' (solo se ESPORTA_HF=True)

print(f'Dominio: {DOMINIO} | Modello: {MODELLO} | Esempi: {N_ESEMPI}')

## 3. Verifica installazione

In [ ]:
!forge list-domains

## 4. Generazione Dataset (Gemini)

In [ ]:
DATASET_PATH = f'data/processed/{DOMINIO}.jsonl'

!forge dataset \
    --dominio {DOMINIO} \
    --n-examples {N_ESEMPI} \
    --output {DATASET_PATH}

In [ ]:
# Anteprima dataset generato
import json
with open(DATASET_PATH) as f:
    esempi = [json.loads(l) for l in f if l.strip()]
print(f'Esempi totali: {len(esempi)}')
print('\n--- Primo esempio ---')
for msg in esempi[0]['messages']:
    print(f"[{msg['role']}] {msg['content'][:120]}")

# Copia dataset su Drive
import shutil
shutil.copy(DATASET_PATH, f'{DRIVE_DIR}/{DOMINIO}.jsonl')
print(f'\n✓ Dataset salvato su Drive → {DRIVE_DIR}/{DOMINIO}.jsonl')

## 5. Training QLoRA

In [ ]:
import uuid
RUN_ID = f'{DOMINIO}-{uuid.uuid4().hex[:6]}'
print(f'Run ID: {RUN_ID}')

!forge train \
    {DATASET_PATH} \
    --dominio {DOMINIO} \
    --modello {MODELLO} \
    --run-id {RUN_ID}

In [ ]:
# Verifica checkpoint
from pathlib import Path
merged_path = Path(f'outputs/checkpoints/{RUN_ID}/merged')
if merged_path.exists():
    files = list(merged_path.iterdir())
    print(f'✓ Merged model: {merged_path}')
    print(f'  File: {[f.name for f in files]}')
else:
    print(f'⚠️  Path non trovato: {merged_path}')

In [ ]:
# Copia modello su Drive
import shutil
drive_model_dir = f'{DRIVE_DIR}/checkpoints/{RUN_ID}'
shutil.copytree(f'outputs/checkpoints/{RUN_ID}', drive_model_dir, dirs_exist_ok=True)
print(f'✓ Modello salvato su Drive → {drive_model_dir}')

## 6. Export

In [ ]:
# Export su HuggingFace Hub (opzionale)
if ESPORTA_HF and HF_REPO:
    !forge export outputs/checkpoints/{RUN_ID}/merged \
        --format hf \
        --repo-id {HF_REPO}
else:
    print('Export HF skippato (ESPORTA_HF=False o HF_REPO vuoto)')

In [ ]:
# Test rapido del modello fine-tunato
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

tokenizer = AutoTokenizer.from_pretrained(str(merged_path))
model = AutoModelForCausalLM.from_pretrained(
    str(merged_path),
    torch_dtype=torch.float16,
    device_map='auto'
)

# Recupera il system prompt dal config del dominio
import yaml
with open(f'config/domains/{DOMINIO}.yaml') as f:
    cfg = yaml.safe_load(f)
system_prompt = cfg.get('dataset', {}).get('obiettivo', 'Sei un assistente utile.')

messages = [
    {'role': 'system', 'content': system_prompt},
    {'role': 'user', 'content': 'Vorrei prenotare un tavolo per sabato sera.'}
]
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(text, return_tensors='pt').to(model.device)

with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=200, temperature=0.7, do_sample=True)

response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
print(f'User: Vorrei prenotare un tavolo per sabato sera.')
print(f'\nAssistant: {response}')

---
## Riepilogo

| Step | Output |
|------|--------|
| Dataset | `data/processed/{dominio}.jsonl` → Drive |
| Modello merged | `outputs/checkpoints/{run_id}/merged/` → Drive |
| HF Hub | opzionale, se `ESPORTA_HF=True` |